# Ensemble Learning: Voting Regressor
---

## Introduction

**Ensemble learning** is a machine learning technique that combines multiple models to produce a stronger and more robust predictor than any individual model alone.

### What is a Voting Regressor?

A **Voting Regressor** is an ensemble meta-estimator that:
- Trains several **base regressors** independently on the same dataset
- Predicts the final output by **averaging** the predictions of all base regressors
- Leverages the strengths of different algorithms to reduce variance and bias

### Dataset: California Housing

We use the **California Housing dataset** from `sklearn.datasets`, which contains:
- **20,640 samples** of California housing blocks
- **8 features**: median income, house age, average rooms, average bedrooms, population, average occupancy, latitude, longitude
- **Target**: Median house value (in $100,000s)

### Objective

Compare the performance of individual regressors (Linear Regression, Decision Tree) against a **Voting Regressor** ensemble using **10-fold cross-validation** and **R² score**.

## Import Libraries

In [2]:
import pandas as pd 
import numpy as np 

# Dataset
from sklearn.datasets import fetch_california_housing

# Individual Regressors
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# Ensemble
from sklearn.ensemble import VotingRegressor

# Evaluation
from sklearn.model_selection import cross_val_score

## Load and Explore the Dataset

In [3]:
# Load the California Housing dataset
df = fetch_california_housing(as_frame=True)

In [4]:
# Preview the feature data
print("Shape:", df.data.shape)
print("\nFeature Names:", df.feature_names)
print("\nFirst 5 rows:")
df.data.head()

Shape: (20640, 8)

Feature Names: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

First 5 rows:


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [5]:
# Statistical summary of features
df.data.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000


##  Prepare Features and Target

In [6]:
# Separate features (X) and target variable (y)
X = df.data      # Features: 8 housing attributes
y = df.target    # Target: Median house value in $100,000s

print(f"Features shape : {X.shape}")
print(f"Target shape   : {y.shape}")
print(f"Target range   : {y.min():.2f} - {y.max():.2f}")

Features shape : (20640, 8)
Target shape   : (20640,)
Target range   : 0.15 - 5.00


## Define Base Regressors

We instantiate four candidate base models:

| Model | Description |
|-------|-------------|
| **Linear Regression** | Fits a linear relationship between features and target |
| **Decision Tree** | Non-linear, rule-based splitting |
| **SVR** | Support Vector Regression with RBF kernel |
| **KNN** | Predicts based on k nearest neighbors |

In [7]:
# Instantiate individual regressors
lr  = LinearRegression()
dt  = DecisionTreeRegressor()
svr = SVR()
knn = KNeighborsRegressor()

## Section 5: Evaluate Individual Models

Before building the ensemble, we benchmark each base model using **10-fold cross-validation** with **R² scoring**.  
R² measures the proportion of variance explained by the model (1.0 = perfect fit).

In [11]:
# Define the estimators to evaluate
estimators = [('lr', lr), ('dt', dt)]

print("Individual Model Performance (10-Fold CV, R² Score)")
print("-" * 45)

individual_scores = {}
for name, model in estimators:
    scores = cross_val_score(model, X, y, cv=10, scoring='r2')
    mean_score = np.round(np.mean(scores), 2)
    individual_scores[name] = mean_score
    print(f"  {name.upper():<25} R² = {mean_score}")

Individual Model Performance (10-Fold CV, R² Score)
---------------------------------------------
  LR                        R² = 0.51
  DT                        R² = 0.23


##  Build and Evaluate the Voting Regressor

The **VotingRegressor** combines the base estimators by averaging their predictions.  
We expect the ensemble to outperform individual models due to the complementary nature of the base learners.

In [12]:
# Build the Voting Regressor with both base models
vr = VotingRegressor(estimators)

# Evaluate using 10-fold cross-validation
scores = cross_val_score(vr, X, y, cv=10, scoring='r2')
ensemble_score = np.round(np.mean(scores), 2)

print("Voting Regressor Performance (10-Fold CV, R² Score)")
print("-" * 45)
print(f"  {'VOTING REGRESSOR':<25} R² = {ensemble_score}")

Voting Regressor Performance (10-Fold CV, R² Score)
---------------------------------------------
  VOTING REGRESSOR          R² = 0.54


##  Compare Results

In [13]:
# Side-by-side comparison of all models
print("Model Comparison (R² Score - Higher is Better)")
print("=" * 45)
for name, score in individual_scores.items():
    print(f"  {name.upper():<25} R² = {score}")
print("-" * 45)
print(f"  {'VOTING REGRESSOR':<25} R² = {ensemble_score}")
print("=" * 45)

Model Comparison (R² Score - Higher is Better)
  LR                        R² = 0.51
  DT                        R² = 0.23
---------------------------------------------
  VOTING REGRESSOR          R² = 0.54


## Summary

In this notebook, we demonstrated the use of a **Voting Regressor** ensemble on the California Housing dataset.

### Results

| Model | R² Score |
|-------|----------|
| Linear Regression (LR) | 0.51 |
| Decision Tree (DT) | 0.23 |
| **Voting Regressor (LR + DT)** | **0.54** |

### Key Takeaways

1. **Individual models performed moderately**: Linear Regression achieved R² = 0.51, while Decision Tree underperformed at R² = 0.23 due to overfitting on training folds.
2. **The Voting Regressor outperformed both**: By averaging predictions, the ensemble achieved R² = 0.54, demonstrating improved generalization.
3. **Diversity matters**: Combining models with different learning strategies (linear vs. tree-based) allows the ensemble to compensate for each model's weaknesses.
4. **Further improvement possible**: Adding more diverse base estimators (e.g., SVR, KNN) or using weighted voting could further boost performance.

### Conclusion

Voting Regressors are a simple yet effective ensemble strategy. Even with just two base learners, the ensemble surpassed both individual models, confirming that **combining diverse models leads to better predictive performance**.